# Naive Bayes Gaussiano desde Cero


## ¿Cuál es el objetivo de Naive Bayes?


El objetivo de **Naive Bayes** es clasificar ejemplos en una clase $y$ usando la probabilidad posterior:

$$P(y\mid x) \propto P(y)\,P(x\mid y)$$

Se le llama *naive* porque asume que las variables de entrada son **condicionalmente independientes** dada la clase.


**Supongamos un conjunto de datos simples**

Usaremos un dataset sintético con 2 clases y 2 variables para visualizar mejor el problema de clasificación.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets

# Dataset sintético (2 clases, 2 features para visualización)
X_vis, y_vis = datasets.make_classification(
    n_samples=300,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    class_sep=1.4,
    random_state=42
)

plt.figure(figsize=(7, 5))
plt.scatter(X_vis[:, 0], X_vis[:, 1], c=y_vis, cmap='coolwarm', alpha=0.8, edgecolor='k')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Datos sintéticos para clasificación')
plt.grid(True)
plt.show()


## ¿Qué clase es la más probable?


En clasificación, no buscamos una recta de ajuste como en regresión, sino la clase con mayor probabilidad posterior.

Para cada muestra $x$, comparamos:

$$\log P(y=c) + \sum_j \log P(x_j\mid y=c)$$

y elegimos la clase con el valor más alto.


In [ ]:
# Visualización de distribuciones de una feature por clase
feature_idx = 0
class_0 = X_vis[y_vis == 0, feature_idx]
class_1 = X_vis[y_vis == 1, feature_idx]

plt.figure(figsize=(7, 5))
plt.hist(class_0, bins=20, alpha=0.6, label='Clase 0', density=True)
plt.hist(class_1, bins=20, alpha=0.6, label='Clase 1', density=True)
plt.xlabel('Feature 1')
plt.ylabel('Densidad')
plt.title('Distribución de Feature 1 por clase')
plt.grid(True)
plt.legend()
plt.show()


Una forma de decidir qué clase es mejor para una muestra es modelar cada feature con una distribución normal por clase y combinar esas probabilidades con el prior de cada clase.


## Demostración Matemática


### Paso 1: Teorema de Bayes

Para una clase $c$ y un vector de características $x=(x_1,\dots,x_d)$:

$$P(y=c\mid x)=\frac{P(x\mid y=c)P(y=c)}{P(x)}$$

Como $P(x)$ es común para todas las clases, basta maximizar $P(x\mid y=c)P(y=c)$.


### Paso 2: Suposición Naive

Asumimos independencia condicional entre features:

$$P(x\mid y=c)=\prod_{j=1}^{d}P(x_j\mid y=c)$$

Para el caso Gaussiano:

$$P(x_j\mid y=c)=\frac{1}{\sqrt{2\pi\sigma_{cj}^2}}\exp\left(-\frac{(x_j-\mu_{cj})^2}{2\sigma_{cj}^2}\right)$$


### Paso 3: Forma logarítmica

Para evitar underflow numérico, se usa logaritmo:

$$\log P(y=c\mid x) \propto \log P(y=c) + \sum_{j=1}^{d}\log P(x_j\mid y=c)$$

Finalmente, predecimos:

$$\hat{y}=\arg\max_c\left[\log P(y=c)+\sum_j\log P(x_j\mid y=c)\right]$$


### Expresión final:

1. Estimar por clase: medias $\mu$, varianzas $\sigma^2$ y priors $P(y=c)$.
2. Evaluar la posterior logarítmica para cada clase.
3. Elegir la clase con mayor score.


### Ejemplo numérico

Si para una muestra $x$ obtenemos:

- Clase 0: $\log P(y=0\mid x) = -5.2$
- Clase 1: $\log P(y=1\mid x) = -3.8$

entonces se predice la **Clase 1**, porque $-3.8 > -5.2$.


## Implementación con Código


### Funciones


In [ ]:
def accuracy(y_true, y_pred):
    return np.sum(y_true == y_pred) / len(y_true)


In [ ]:
import numpy as np

class NaiveBayes:
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self._classes = np.unique(y)
        n_classes = len(self._classes)

        # calculate mean, var, and prior for each class
        self._mean = np.zeros((n_classes, n_features), dtype=np.float64)
        self._var = np.zeros((n_classes, n_features), dtype=np.float64)
        self._priors = np.zeros(n_classes, dtype=np.float64)

        for idx, c in enumerate(self._classes):
            X_c = X[y == c]
            self._mean[idx, :] = X_c.mean(axis=0)
            self._var[idx, :] = X_c.var(axis=0)
            self._priors[idx] = X_c.shape[0] / float(n_samples)

    def predict(self, X):
        y_pred = [self._predict(x) for x in X]
        return np.array(y_pred)

    def _predict(self, x):
        posteriors = []

        # calculate posterior probability for each class
        for idx, c in enumerate(self._classes):
            prior = np.log(self._priors[idx])
            posterior = np.sum(np.log(self._pdf(idx, x)))
            posterior = prior + posterior
            posteriors.append(posterior)

        # return class with highest posterior probability
        return self._classes[np.argmax(posteriors)]

    def _pdf(self, class_idx, x):
        mean = self._mean[class_idx]
        var = self._var[class_idx] + 1e-9
        numerator = np.exp(-((x - mean) ** 2) / (2 * var))
        denominator = np.sqrt(2 * np.pi * var)
        return numerator / denominator


### Clase


In [ ]:
# Testing en estilo notebook (equivalente al bloque __main__)
from sklearn.model_selection import train_test_split
from sklearn import datasets

X, y = datasets.make_classification(
    n_samples=1000, n_features=10, n_classes=2, random_state=123
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123
)

nb = NaiveBayes()
nb.fit(X_train, y_train)
predictions = nb.predict(X_test)

print('Naive Bayes classification accuracy', accuracy(y_test, predictions))


In [ ]:
# Predicción rápida de nuevas muestras
new_samples = X_test[:5]
print('Muestras:', new_samples.shape)
print('Predicciones:', nb.predict(new_samples))
print('Etiquetas reales:', y_test[:5])
